# Module 13: UI & Observability

## What You'll Learn

- Feast Web UI (Beta): what it shows, how to access, limitations
- Marquez Lineage UI: lineage graphs, job history
- RHOAI Dashboard integration (current and planned)
- Prometheus metrics: `feast_feature_server_request_latency`, `feast_feature_freshness`, etc.
- ServiceMonitor auto-generation (v0.62.0)
- Building a Grafana dashboard for Feast
- Future: Data Hub UI vision (Workshop Decision #3)

## Prerequisites

- Feast deployed via operator (Module 12)
- `oc` CLI authenticated to your cluster
- OpenShift monitoring enabled (for Prometheus metrics)

---

> **🖥️ UI**: This module is ALL about UIs — Feast UI, Marquez, RHOAI Dashboard, Grafana. We bring them together so you know where to look for what.
>
> **📍 RHOAI STATUS**: Feast UI available (Beta). Prometheus metrics available. ServiceMonitor auto-generated (v0.62.0+).
>
> **🗺️ DATA STRATEGY**: Workshop Decision #3 — **Data Hub UI in AI Hub**. The vision is unified discovery over all data registries (Feast, MLflow, OGX, connections). Today, each UI is separate.

## UI Landscape Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                    Feast Observability Stack                     │
│                                                                  │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────────────┐ │
│  │  Feast UI    │  │ Marquez UI   │  │  RHOAI Dashboard     │ │
│  │  (Beta)      │  │ (separate)   │  │  (GA)                │ │
│  │              │  │              │  │                      │ │
│  │ Feature defs │  │ Lineage DAG  │  │ Projects, workbenches│ │
│  │ Entities     │  │ Job history  │  │ Model registry       │ │
│  │ Data sources │  │ Dataset ver. │  │ Data Hub (future)    │ │
│  └──────────────┘  └──────────────┘  └──────────────────────┘ │
│                                                                  │
│  ┌──────────────────────────────────────────────────────────┐   │
│  │  Prometheus / Grafana (metrics)                           │   │
│  │  feast_feature_freshness, feast_feature_server_latency  │   │
│  └──────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────┘
```

> **⚠️ GAP**: No unified Data Hub view — Feast UI, MLflow UI, Marquez UI are all separate apps with no cross-linking.
>
> **⚠️ GAP**: No alerting when materialization fails or features go stale beyond TTL.

## 1. Feast Web UI (Beta)

A React-based interface for browsing feature store metadata. Read-only — you cannot create or modify features from the UI.

### What You Can See

| Section | Content |
|---------|--------|
| Feature Views | Schema, entities, TTL, data sources |
| Entities | Join keys, value types |
| Data Sources | Batch and stream source definitions |
| Feature Services | Grouped features for serving |
| Saved Datasets | Profiled training datasets |

> **⚠️ GAP**: Feast UI is **Beta** — read-only, cannot manage permissions ([Issue #6192](https://github.com/feast-dev/feast/issues/6192)).
>
> **📍 RHOAI STATUS**: Available when `spec.services.ui: {}` is set in the FeatureStore CR (Module 12).

In [ ]:
# Access Feast UI on RHOAI

feast_ui_commands = """
# List Feast routes in your namespace
oc get routes -n my-ds-project | grep feast

# Typical route names:
#   feast-ui-my-ds-project.apps.<cluster-domain>
#   feast-feature-server-my-ds-project.apps.<cluster-domain>

# Get the exact URL
oc get route feast-ui -n my-ds-project -o jsonpath='https://{.spec.host}{"\\n"}'

# Alternative: port-forward for local access
oc port-forward svc/feast-ui 8888:8888 -n my-ds-project
# Then open http://localhost:8888
"""
print(feast_ui_commands)

## 2. Marquez Lineage UI

When OpenLineage is enabled (Module 07), Feast emits lineage events during `feast apply` and `feast materialize`. Marquez collects these and provides a visual DAG.

### What You Can See

- **Lineage Graph**: data sources → feature views → feature services
- **Job History**: materialization runs, duration, status
- **Dataset Versions**: schema evolution over time
- **Cross-System Lineage**: if Spark/KFP also emit to the same Marquez instance

> **⚠️ GAP**: Lineage stops at the feature store boundary — no tracking into model training or serving.
>
> **⚠️ GAP**: Marquez is not bundled in RHOAI — deploy separately for POC evaluation.

In [ ]:
# Access Marquez Lineage UI

marquez_commands = """
# Check if Marquez is deployed
oc get routes -n my-ds-project | grep marquez
oc get pods -n my-ds-project | grep marquez

# Port-forward Marquez web UI (default port 3000)
oc port-forward svc/marquez-web 3000:3000 -n my-ds-project
# Open http://localhost:3000

# Query lineage via Marquez API
curl -s http://localhost:5000/api/v1/namespaces/feast/jobs | jq '.jobs[].name'
"""
print(marquez_commands)

## 3. RHOAI Dashboard (AI Hub)

The main RHOAI web console. Feast-relevant sections today and planned:

| Section | What You See | Status |
|---------|-------------|--------|
| Data Science Projects | Namespace where Feast is deployed | GA |
| Workbenches | Run notebooks that use Feast | GA |
| Model Registry | Models trained with Feast features | GA |
| Data Connections | S3/storage (not yet Feast-aware) | GA (limited) |
| **Data Hub** (future) | Unified: features, datasets, vectors, connections | Planned |

### Future: Data Hub UI (Workshop Decision #3)

The vision for AI Hub's "Data Assets" section:

- Feast Registry (feature views, entities)
- OGX Sources Registry (vector stores, documents)
- MLflow Registry (experiments, models)
- External Connections
- Unified Lineage Visualization

> **🗺️ DATA STRATEGY**: Workshop Decision #3 — Data Hub UI in AI Hub. Unified discovery over all data registries. This is future state; today you navigate separate UIs.

In [ ]:
# Access RHOAI Dashboard and related UIs

dashboard_commands = """
# RHOAI Dashboard URL
oc get routes -n redhat-ods-applications | grep rhods-dashboard

# MLflow UI (if deployed in your project)
oc get routes -n my-ds-project | grep mlflow

# All application routes in your DS project
oc get routes -n my-ds-project
"""
print(dashboard_commands)

## 4. Prometheus Metrics

Feast exposes native Prometheus metrics (v0.61.0+). On OpenShift, the operator auto-generates a **ServiceMonitor** (v0.62.0+) for automatic scraping.

### Available Metrics

| Metric | What It Measures |
|--------|-----------------|
| `feast_feature_server_request_latency` | Feature serving response time (histogram) |
| `feast_feature_freshness` | Time since last materialization per feature view |
| `feast_materialization_duration` | How long materialization jobs take |
| `feast_odfv_transformation_duration` | On-Demand Feature View compute time |

> **📍 RHOAI STATUS**: Prometheus metrics available. ServiceMonitor auto-generated.
>
> **⚠️ GAP**: No pre-built Grafana dashboard for Feast in RHOAI.
>
> **🔮 UPSTREAM**: Go server metrics and HTTP/gRPC tracing added in v0.61.0.

In [ ]:
# Verify ServiceMonitor and scrape targets

prometheus_setup = """
# Check ServiceMonitor exists (auto-generated by operator v0.62.0+)
oc get servicemonitor -n my-ds-project | grep feast

# Inspect ServiceMonitor spec
oc get servicemonitor feast-feature-server -n my-ds-project -o yaml

# Port-forward to cluster Prometheus (requires monitoring admin access)
oc port-forward svc/prometheus-k8s 9090:9090 -n openshift-monitoring

# Or use OpenShift Console: Observe → Metrics
"""
print(prometheus_setup)

In [ ]:
# Example PromQL queries for feature freshness monitoring

promql_queries = {
    "feature_freshness_seconds": (
        'feast_feature_freshness{namespace="my-ds-project"}'
    ),
    "stale_features_alert": (
        'feast_feature_freshness{namespace="my-ds-project"} > 86400'  # > 24 hours
    ),
    "serving_latency_p99": (
        'histogram_quantile(0.99, '
        'rate(feast_feature_server_request_latency_bucket{namespace="my-ds-project"}[5m]))'
    ),
    "serving_latency_p50": (
        'histogram_quantile(0.50, '
        'rate(feast_feature_server_request_latency_bucket{namespace="my-ds-project"}[5m]))'
    ),
    "materialization_duration": (
        'feast_materialization_duration{namespace="my-ds-project"}'
    ),
    "odfv_compute_time": (
        'feast_odfv_transformation_duration{namespace="my-ds-project"}'
    ),
}

print("PromQL Queries for Feast Monitoring:\n")
for name, query in promql_queries.items():
    print(f"# {name}")
    print(query)
    print()

In [ ]:
# Query Prometheus programmatically (if accessible from workbench)
import requests
import json

def query_prometheus(prom_url, promql, timeout=10):
    """Execute a PromQL query against Prometheus."""
    try:
        resp = requests.get(
            f"{prom_url}/api/v1/query",
            params={"query": promql},
            timeout=timeout,
            verify=False  # cluster-internal; adjust for your environment
        )
        resp.raise_for_status()
        data = resp.json()
        results = data.get("data", {}).get("result", [])
        if not results:
            print("No data returned (metrics may not be scraped yet)")
            return []
        for r in results:
            labels = r.get("metric", {})
            value = r.get("value", [None, None])[1]
            fv = labels.get("feature_view", labels.get("job", "unknown"))
            print(f"  {fv}: {value}")
        return results
    except requests.exceptions.ConnectionError:
        print("Prometheus not reachable — use port-forward or OpenShift Console")
        print("  oc port-forward svc/prometheus-k8s 9090:9090 -n openshift-monitoring")
        return []
    except Exception as e:
        print(f"Query failed: {e}")
        return []

# Example query (uncomment with live Prometheus access)
# query_prometheus("http://localhost:9090", promql_queries["feature_freshness_seconds"])
print("Run query_prometheus() after port-forwarding to Prometheus")

## 5. Building a Grafana Dashboard

There is no OOTB Grafana dashboard for Feast in RHOAI. Build your own using the PromQL queries above.

### Recommended Panels

| Panel | Query | Purpose |
|-------|-------|--------|
| Feature Freshness | `feast_feature_freshness` | Time since last materialize per FV |
| Serving Latency P99 | `histogram_quantile(0.99, ...)` | SLA monitoring |
| Materialization Duration | `feast_materialization_duration` | Job health |
| ODFV Compute Time | `feast_odfv_transformation_duration` | Real-time transform perf |

> **⚠️ GAP**: No alerting when materialization fails or features go stale. You must configure Alertmanager rules manually.

In [ ]:
# Grafana dashboard JSON skeleton for Feast
import json

grafana_dashboard = {
    "title": "Feast Feature Store",
    "tags": ["feast", "feature-store", "rhoai"],
    "timezone": "browser",
    "panels": [
        {
            "title": "Feature Freshness (seconds since last materialize)",
            "type": "timeseries",
            "gridPos": {"h": 8, "w": 12, "x": 0, "y": 0},
            "targets": [{
                "expr": 'feast_feature_freshness{namespace="$namespace"}',
                "legendFormat": "{{feature_view}}"
            }]
        },
        {
            "title": "Serving Latency P99",
            "type": "timeseries",
            "gridPos": {"h": 8, "w": 12, "x": 12, "y": 0},
            "targets": [{
                "expr": (
                    'histogram_quantile(0.99, '
                    'rate(feast_feature_server_request_latency_bucket{namespace="$namespace"}[5m]))'
                ),
                "legendFormat": "p99"
            }]
        },
        {
            "title": "Materialization Duration",
            "type": "timeseries",
            "gridPos": {"h": 8, "w": 12, "x": 0, "y": 8},
            "targets": [{
                "expr": 'feast_materialization_duration{namespace="$namespace"}',
                "legendFormat": "{{feature_view}}"
            }]
        },
        {
            "title": "Stale Features (>24h)",
            "type": "stat",
            "gridPos": {"h": 4, "w": 6, "x": 12, "y": 8},
            "targets": [{
                "expr": 'count(feast_feature_freshness{namespace="$namespace"} > 86400)',
            }]
        }
    ],
    "templating": {
        "list": [{
            "name": "namespace",
            "type": "query",
            "query": "label_values(feast_feature_freshness, namespace)"
        }]
    }
}

with open("feast-grafana-dashboard.json", "w") as f:
    json.dump(grafana_dashboard, f, indent=2)

print("Created feast-grafana-dashboard.json")
print("Import via Grafana UI: Dashboards → Import → Upload JSON")

## 6. Where to Look — Quick Reference

| I want to... | Go to... | Status |
|--------------|----------|--------|
| Browse feature definitions | Feast UI | Available (Beta) |
| See data lineage graph | Marquez UI | Deploy separately |
| Manage RHOAI resources | RHOAI Dashboard | GA |
| Monitor feature freshness | Prometheus/Grafana | Metrics available, no OOTB dashboard |
| See model → feature relationships | Not available | Future (Data Hub) |
| Manage Feast RBAC permissions | Not available (code only) | Issue #6192 |

For full details on each UI, see **[docs/ui-guide.md](../../docs/ui-guide.md)**.

In [ ]:
# One-shot: discover all Feast-related UIs in your namespace

discover_uis = """
# Run this to find all accessible UIs:
echo "=== Feast Routes ==="
oc get routes -n my-ds-project -o custom-columns=NAME:.metadata.name,HOST:.spec.host | grep -E 'feast|marquez|mlflow'

echo "=== ServiceMonitors ==="
oc get servicemonitor -n my-ds-project | grep feast

echo "=== RHOAI Dashboard ==="
oc get route rhods-dashboard -n redhat-ods-applications -o jsonpath='https://{.spec.host}{"\\n"}'
"""
print(discover_uis)
print("\nFull UI reference: docs/ui-guide.md")

## Summary

| UI | Purpose | RHOAI Status |
|----|---------|-------------|
| Feast UI | Browse feature metadata | Beta, operator-managed |
| Marquez UI | Lineage graphs | Deploy separately |
| RHOAI Dashboard | Platform management | GA |
| Prometheus/Grafana | Metrics & alerting | Metrics GA, dashboard DIY |
| Data Hub (future) | Unified discovery | Workshop Decision #3 |

**Next**: [Module 14: Credit Scoring Scenario](../14-scenario-credit-scoring/14-scenario-credit-scoring.ipynb) — end-to-end predictive AI with Feast.